# EHR Feature Extraction - Vitals & Labs
## Extracting Clinical Measurements Aligned to Pregnancy Timeline

**Purpose:** Extract vital signs and laboratory measurements from EHR data, organized by pregnancy time periods to KP model training features.

**Time periods:**
- **Pre-pregnancy (pp_):** All measurements before estimated LMP (up to 2 years prior)
- **Early pregnancy (ep_):** From LMP to 12 weeks gestation
- **Late pregnancy (lp_):** From 12 weeks to delivery (excluded in 12-week models)


File is commented out because it builds a dataset in the All of Us open research access dataset. If you have an account, you can uncomment the code to run and duplicate our cohort. 

# Setup & Configuration

In [ ]:
'''
from IPython.display import display, HTML
import pandas as pd
import numpy, scipy
import matplotlib, matplotlib.pyplot as plt
from matplotlib import rcParams
from datetime import date
import seaborn as sns
import os
import subprocess
import pickle
import numpy as np

my_bucket = os.getenv('WORKSPACE_BUCKET')
data_subfolder = 'pregnancy_cohort_1'
print(my_bucket)
'''

In [ ]:
'''
version = %env WORKSPACE_CDR
version
'''

# Load pregnancy cohort

In [ ]:
'''
df_final = pd.read_pickle("./pregnancy_episodes_date.pkl")
df_final
'''

In [ ]:
'''
def get_top_freq(df, id_name, int_cutoff = 25):
    vc = df[id_name].value_counts()
    return df[df[id_name].isin(vc[:int_cutoff].index)]

def custom_sql(person_ids, sql_command, num_splits = 5):
   
    CDR = os.environ['WORKSPACE_CDR']

    div_index = len(person_ids) // num_splits

    for i in range(num_splits + 1): 
        ids_string = ', '.join(str(e) for e in person_ids[i*div_index:(i + 1)*div_index])
        if len(ids_string) == 0:
            continue
        
        df_temp = pd.read_gbq(sql_command.format(CDR, ids_string),
            use_bqstorage_api=("BIGQUERY_STORAGE_API_ENABLED" in os.environ),
            dialect = "standard", progress_bar_type="tqdm_notebook")

        if i == 0: 
            df = df_temp 
        else: 
            df = pd.concat([df, df_temp])
    return df

def compute_summary_statistics(df, concept_name, cutoff_name ='all_', stats = []): 
    concept_id = sorted(pd.unique(df[concept_name]))
    df_binary = pd.DataFrame(index = pd.unique(df.index))
    
    if "avg" in stats: 
        for p in concept_id:
            temp = df[df[concept_name] == p].groupby("id")['VALUE_AS_NUMBER'].mean().round(3)
            df_binary[cutoff_name + str(p) + "_avg"] = temp


    if "min" in stats: 
        for p in concept_id:
            temp = df[df[concept_name] == p].groupby("id")['VALUE_AS_NUMBER'].min().round(3)
            df_binary[cutoff_name + str(p) + "_min"] = temp
    
    if "max" in stats: 
        for p in concept_id:
            temp = df[df[concept_name] == p].groupby("id")['VALUE_AS_NUMBER'].max().round(3)
            df_binary[cutoff_name + str(p) + "_max"] = temp
    
    if "std" in stats:
         for p in concept_id: 
            temp = df[df[concept_name] == p].groupby("id")['VALUE_AS_NUMBER'].std().round(3)
            df_binary[cutoff_name + str(p) + "_std"] = temp

    return df_binary
'''

## Demographics

In [ ]:
'''
DATASET = os.getenv('WORKSPACE_CDR')
'''

In [ ]:
'''
person_ids = list(set(df_final['person_id']))
person_ids_sub = tuple(set(df_final['person_id']))
'''

In [ ]:
'''
query = f"""
        SELECT
            person.person_id,
            DATE(person.birth_datetime) AS birth_date,
            p_race_concept.concept_name as race,
            p_gender_concept.concept_name as gender,
            p_ethnicity_concept.concept_name as ethnicity,
            p_sex_at_birth_concept.concept_name as sex_at_birth 
        FROM  `{DATASET}.person` person 
        LEFT JOIN `{DATASET}.concept` p_race_concept 
                ON person.race_concept_id = p_race_concept.concept_id
        LEFT JOIN `{DATASET}.concept` p_gender_concept 
                ON person.gender_concept_id = p_gender_concept.concept_id
        LEFT JOIN `{DATASET}.concept` p_ethnicity_concept 
                ON person.ethnicity_concept_id = p_ethnicity_concept.concept_id 
        LEFT JOIN `{DATASET}.concept` p_sex_at_birth_concept 
                ON person.sex_at_birth_concept_id = p_sex_at_birth_concept.concept_id 
        WHERE person_id IN {person_ids_sub} --only person_ids in df_cohort
    """
df_person = pd.read_gbq(query, dialect="standard")
'''

In [ ]:
'''
df_final['person_id'] = df_final['person_id'].astype(np.int64)
'''

In [ ]:
'''
df_final = df_final.merge(df_person, on=['person_id'], how='left')
'''

In [ ]:
'''
print(df_final)
df_final.to_csv('pregnancy_demographics_date.csv')
'''

In [ ]:
'''
df_merge = df_final[['person_id', 'censor_date', 'id']].copy()
'''

## Labs

In [ ]:
'''
custom_chem_meas_id = "3000905, 3000963, 3004501, 3006906, 3006923, 3009744, 3012030, 3013650, 3013682, 3013721, 3014576, 3015632, 3016723, 3019550, 3019897"
'''

In [ ]:
'''
sql_command_chem_meas = """
SELECT 
        measurement.MEASUREMENT_CONCEPT_ID,
        measurement.MEASUREMENT_DATETIME,
        measurement.PERSON_ID,
        measurement.VALUE_AS_NUMBER,
        measurement.VISIT_OCCURRENCE_CONCEPT_NAME,
        m_unit.concept_name as UNIT_CONCEPT_NAME,
        m_standard_concept.concept_name as STANDARD_CONCEPT_NAME 
FROM 
    (SELECT * 
    FROM `{0}.ds_measurement` measurement  
    WHERE 
        person_id IN ({1})
        AND measurement_concept_id in  (
                    select
                        distinct c.concept_id 
                    from
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    join
                        (
                            select
                                cast(cr.id as string) as id 
                            from
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr 
                            where
                                domain_id = 'MEASUREMENT' 
                                and is_standard = 1 
                                and concept_id in (
                                """ + custom_chem_meas_id + """
                                )
                                and is_selectable = 1 
                                and full_text like '%[measurement_rank1]%'
                        ) a 
                            on (
                                c.path like concat('%.',
                            a.id,
                            '.%') 
                            or c.path like concat('%.',
                            a.id) 
                            or c.path like concat(a.id,
                            '.%') 
                            or c.path = a.id) 
                        where
                            domain_id = 'MEASUREMENT' 
                            and is_standard = 1 
                            and is_selectable = 1
                        )
    ) measurement
LEFT JOIN
    `{0}.concept` m_unit 
        on measurement.unit_concept_id = m_unit.concept_id
LEFT JOIN
    `{0}.concept` m_standard_concept 
                on measurement.measurement_concept_id = m_standard_concept.concept_id

"""
df_chem_meas = custom_sql(person_ids, sql_command_chem_meas, num_splits = 3)
df_chem_meas.head(5)
'''

In [ ]:
'''
#change all variables in labs to match kp names
df_chem_meas['MEASUREMENT_CONCEPT_ID'] = df_chem_meas['MEASUREMENT_CONCEPT_ID'].astype(str)

df_chem_meas['MEASUREMENT_CONCEPT_ID'] = df_chem_meas['MEASUREMENT_CONCEPT_ID'].replace('3000905', 'WBC')
df_chem_meas['MEASUREMENT_CONCEPT_ID'] = df_chem_meas['MEASUREMENT_CONCEPT_ID'].replace('3000963', 'HGB')
df_chem_meas['MEASUREMENT_CONCEPT_ID'] = df_chem_meas['MEASUREMENT_CONCEPT_ID'].replace('3004501', 'GLU_RAN')
df_chem_meas['MEASUREMENT_CONCEPT_ID'] = df_chem_meas['MEASUREMENT_CONCEPT_ID'].replace('3006906', 'CALCIUM')
df_chem_meas['MEASUREMENT_CONCEPT_ID'] = df_chem_meas['MEASUREMENT_CONCEPT_ID'].replace('3006923', 'ALT')

df_chem_meas['MEASUREMENT_CONCEPT_ID'] = df_chem_meas['MEASUREMENT_CONCEPT_ID'].replace('3009744', 'RBC')
df_chem_meas['MEASUREMENT_CONCEPT_ID'] = df_chem_meas['MEASUREMENT_CONCEPT_ID'].replace('3013650', 'ANC')
df_chem_meas['MEASUREMENT_CONCEPT_ID'] = df_chem_meas['MEASUREMENT_CONCEPT_ID'].replace('3013682', 'BUN')

df_chem_meas['MEASUREMENT_CONCEPT_ID'] = df_chem_meas['MEASUREMENT_CONCEPT_ID'].replace('3013721', 'AST')
df_chem_meas['MEASUREMENT_CONCEPT_ID'] = df_chem_meas['MEASUREMENT_CONCEPT_ID'].replace('3014576', 'CHLORIDE')
df_chem_meas['MEASUREMENT_CONCEPT_ID'] = df_chem_meas['MEASUREMENT_CONCEPT_ID'].replace('3016723', 'CREATININE')

print(df_chem_meas.shape)
'''

In [ ]:
'''
#change unit of time instead of a censor date to time
df_chem_meas_clean = df_chem_meas
df_chem_meas_clean.rename(columns={'PERSON_ID': 'person_id'}, inplace=True)
df_chem_meas_clean = df_merge.merge(df_chem_meas_clean, on='person_id', how='inner')
print(df_chem_meas_clean.shape)

#negative means pre preg
df_chem_meas_clean['time'] = ((df_chem_meas_clean['MEASUREMENT_DATETIME'] - df_chem_meas_clean['censor_date']).dt.days / 7).round().astype(int)
print(df_chem_meas_clean.head())
'''

In [ ]:
'''
labs = df_chem_meas_clean
labs = labs.set_index('id')
'''

In [ ]:
'''
#make dfs where the time is in the right bucket first
cutoff = 36
censor_date = 0
labs_dfs = []
labs_clean_all = labs[(labs['time'] < censor_date) & (labs['time'] > -104)]

print(labs_clean_all.head())
labs_dfs.append(labs_clean_all)

#if 6 or 12 weeks, make bucket censor - cutoff
#if 18-24, bucket censor-12, 12-cutoff
#if 30, censor-12, 12-24, 24-cutoff
#if 36, censor-12, 12-24, 24-30, 36
labs_after = labs[labs['time'] >= censor_date]
if (cutoff <= 24) and (cutoff > 0):
    labs_dfs.append(labs_after[labs_after['time'] <= cutoff])            
else:
    labs_dfs.append(labs_after[labs_after['time'] <= 24])
    labs_dfs.append(labs_after[(labs_after['time'] > 24) & (labs_after['time'] <= cutoff)]) 
'''

In [ ]:
'''
p_labels = ['pp_', 'ep_', 'lp_']
for i in range(len(labs_dfs)):
    labs_dfs[i] = compute_summary_statistics(labs_dfs[i], 
                           concept_name = 'MEASUREMENT_CONCEPT_ID', 
                           cutoff_name = p_labels[i],
                            stats = ["avg", "min", "max", "std"])
    labs_dfs[i].index.name = 'id'
'''

In [ ]:
'''
#Merge buckets back together
labs_all = pd.concat(labs_dfs, axis = 1, join = 'outer')
labs_all.index.name = 'id'
'''

In [ ]:
'''
print(labs_all.shape)
'''

In [ ]:
'''
print(labs_all.head())
'''

In [ ]:
'''
labs_all.to_csv('labs_hold_36w_date.csv')
'''

## Physical Measurements

In [ ]:
'''
sql_command_phys_meas = """
SELECT 
        measurement.MEASUREMENT_CONCEPT_ID,
        measurement.MEASUREMENT_DATETIME,
        measurement.PERSON_ID,
        measurement.VALUE_AS_NUMBER,
        measurement.VISIT_OCCURRENCE_CONCEPT_NAME,
        m_unit.concept_name as UNIT_CONCEPT_NAME,
        m_standard_concept.concept_name as STANDARD_CONCEPT_NAME
FROM 
    (SELECT * 
    FROM `{0}.ds_measurement` measurement 
    WHERE 
        person_id IN ({1})
        AND measurement_concept_id in  (
                    select
                        distinct c.concept_id 
                    from
                        `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` c 
                    join
                        (
                            select
                                cast(cr.id as string) as id 
                            from
                                `""" + os.environ["WORKSPACE_CDR"] + """.cb_criteria` cr 
                            where
                                domain_id = 'MEASUREMENT' 
                                and is_standard = 1 
                                and concept_id in (
                                    3027018, 3012888, 3004249, 3025315, 40762636
                                ) 
                                and is_selectable = 1 
                                and full_text like '%[measurement_rank1]%'
                        ) a 
                            on (
                                c.path like concat('%.',
                            a.id,
                            '.%') 
                            or c.path like concat('%.',
                            a.id) 
                            or c.path like concat(a.id,
                            '.%') 
                            or c.path = a.id) 
                        where
                            domain_id = 'MEASUREMENT' 
                            and is_standard = 1 
                            and is_selectable = 1
                        )
    ) measurement
LEFT JOIN
    `{0}.concept` m_unit 
        on measurement.unit_concept_id = m_unit.concept_id
LEFT JOIN
    `{0}.concept` m_standard_concept 
            on measurement.measurement_concept_id = m_standard_concept.concept_id

"""

df_phys_meas = custom_sql(person_ids, sql_command_phys_meas, num_splits = 3)
df_phys_meas.head(5)
'''

In [ ]:
'''
df_phys_meas['MEASUREMENT_CONCEPT_ID'] = df_phys_meas.loc[:, 'MEASUREMENT_CONCEPT_ID'].astype(str)

bmi_hold = df_phys_meas[df_phys_meas['MEASUREMENT_CONCEPT_ID'] == '40762636'].copy()

bmi_hold.rename(columns={'PERSON_ID': 'person_id'}, inplace=True)
bmi_hold = df_merge.merge(bmi_hold, on='person_id', how='inner')
print(bmi_hold.shape)
#negative means pre preg
bmi_hold['time'] = ((bmi_hold['MEASUREMENT_DATETIME'] - bmi_hold['censor_date']).dt.days / 7).round().astype(int)

bmi_hold = bmi_hold[bmi_hold['time'] < 0]
bmi_hold = bmi_hold.loc[bmi_hold.groupby('id')['time'].idxmax()]

bmi_hold.rename(columns={'VALUE_AS_NUMBER': 'PPREG_BMI'}, inplace=True)

bmi = bmi_hold[['id', 'PPREG_BMI']].copy()
'''

In [ ]:
'''
# 3025315, 40762636 --> BMI
#change all variables in labs to match kp names

df_phys_meas['MEASUREMENT_CONCEPT_ID'] = df_phys_meas['MEASUREMENT_CONCEPT_ID'].replace('3027018', 'hr')
df_phys_meas['MEASUREMENT_CONCEPT_ID'] = df_phys_meas['MEASUREMENT_CONCEPT_ID'].replace('3012888', 'dbp')
df_phys_meas['MEASUREMENT_CONCEPT_ID'] = df_phys_meas['MEASUREMENT_CONCEPT_ID'].replace('3004249', 'sbp')

print(df_phys_meas.shape)
'''

In [ ]:
'''
#change unit of time instead of a censor date to time
df_phys_meas_clean = df_phys_meas
df_phys_meas_clean.rename(columns={'PERSON_ID': 'person_id'}, inplace=True)
df_phys_meas_clean = df_merge.merge(df_phys_meas_clean, on='person_id', how='inner')
print(df_phys_meas_clean.shape)
#negative means pre preg
df_phys_meas_clean['time'] = ((df_phys_meas_clean['MEASUREMENT_DATETIME'] - df_phys_meas_clean['censor_date']).dt.days / 7).round().astype(int)
print(df_phys_meas_clean.head())
'''

In [ ]:
'''
vitals = df_phys_meas_clean
vitals = vitals.set_index('id')
'''

In [ ]:
'''
#make dfs where the time is in the right bucket first
cutoff = 36
censor_date = 0
vitals_dfs = []
vitals_clean_all = vitals[(vitals['time'] < censor_date) & (vitals['time'] > -104)]

print(vitals_clean_all.head())
vitals_dfs.append(vitals_clean_all)

#if 6 or 12 weeks, make bucket censor - cutoff
#if 18-24, bucket censor-12, 12-cutoff
#if 30, censor-12, 12-24, 24-cutoff
#if 36, censor-12, 12-24, 24-30, 36
vitals_after = vitals[vitals['time'] >= censor_date]
if (cutoff <= 24) and (cutoff > 0):
    vitals_dfs.append(vitals_after[vitals_after['time'] <= cutoff])            
else:
    vitals_dfs.append(vitals_after[vitals_after['time'] <= 24])
    vitals_dfs.append(vitals_after[(vitals_after['time'] > 24) & (vitals_after['time'] <= cutoff)]) 
'''

In [ ]:
'''
p_labels = ['pp_', 'ep_', 'lp_']
for i in range(len(vitals_dfs)):
    vitals_dfs[i] = compute_summary_statistics(vitals_dfs[i], 
                           concept_name = 'MEASUREMENT_CONCEPT_ID', 
                           cutoff_name = p_labels[i],
                            stats = ["avg", "min", "max", "std"])
    vitals_dfs[i].index.name = 'id'
'''

In [ ]:
'''
#Merge buckets back together
vitals_all = pd.concat(vitals_dfs, axis = 1, join = 'outer')
vitals_all.index.name = 'id'
'''

In [ ]:
'''
print(vitals_all.shape)
vitals_all = vitals_all.merge(bmi, how='left', on='id')
print(vitals_all.shape)
print(vitals_all.head())
'''

In [ ]:
'''
print(vitals_all.columns.tolist())
'''

In [ ]:
'''
vitals_all.to_csv('vitals_hold_36w_date.csv')
'''